In [1]:
import joblib
import numpy as np
import pandas as pd

In [2]:
# 1. LOAD PIPELINE ASSETS
# ==========================================
payload = joblib.load('svm_ckd_production_pipeline.pkl')

model = payload['model']
threshold = payload['threshold']
scaler = payload['scaler']
binary_mappings = payload['binary_mappings']
binary_cols = payload['binary_cols']
expected_columns = payload['expected_columns']

print("--- Production Pipeline Successfully Loaded ---")

--- Production Pipeline Successfully Loaded ---


In [3]:
# 2. RECEIVE RAW DATA FROM THE FIELD
# ==========================================
# Dict key names must precisely match your original CSV column definitions
raw_patient_data = {
    'age': 48.0, 'bp': 80.0, 'sg': 'c', 'al': 1.0, 'su': 0.0,
    'rbc': 'normal', 'pc': 'normal', 'pcc': 'notpresent', 'ba': 'notpresent',
    'bgr': 121.0, 'bu': 36.0, 'sc': 1.2, 'sod': 135.0, 'pot': 4.2,
    'hrmo': 15.4, 'pcv': 44.0, 'wc': 7800.0, 'rc': 5.2,
    'htn': 'no', 'dm': 'no', 'cad': 'no', 'appet': 'good', 'pe': 'no', 'ane': 'no'
}

raw_df = pd.DataFrame([raw_patient_data])

In [4]:
# 3. ROBUST PREPROCESSING PIPELINE
# ==========================================

# A. Handle Binary Mapping
raw_df[binary_cols] = raw_df[binary_cols].replace(binary_mappings)

# B. Handle One-Hot Encoding for multi-category fields securely
# Runs get_dummies on the single row entry
processed_df = pd.get_dummies(raw_df, columns=['sg'], dtype=int, drop_first=False)

# C. Enforce Structural Alignment
# Dynamically aligns dataframe layout to mirror the training columns exactly.
# Skips missing categories (e.g., 'sg_e') safely by filling missing shapes with 0.
for col in expected_columns:
    if col not in processed_df.columns:
        processed_df[col] = 0

# Sort columns into the precise order expected by the scaler
final_features_df = processed_df[expected_columns]

# D. Apply original training scale configurations
scaled_data = scaler.transform(final_features_df)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_10740\3761350588.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  raw_df[binary_cols] = raw_df[binary_cols].replace(binary_mappings)


In [5]:
# 4. PREDICT VIA OPTIMIZED THRESHOLD
# ==========================================
# Retrieve decision probability vector for Class 1 (Diseased)
disease_probability = model.predict_proba(scaled_data)[:, 1]

# Apply the perfect 0.42 threshold classification rule
final_prediction = int(disease_probability >= threshold)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_10740\2247523673.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  final_prediction = int(disease_probability >= threshold)


In [6]:
# 5. GENERATE DIAGNOSTIC STATUS REPORT
# ==========================================
print("\n--- Clinical Diagnostic Evaluation ---")
print(f"Calculated Illness Risk: {disease_probability[0]:.4%}")

if final_prediction == 1:
    print("Diagnostic Result: Chronic Kidney Disease DETECTED")
else:
    print("Diagnostic Result: No Disease Detected (Healthy)")


--- Clinical Diagnostic Evaluation ---
Calculated Illness Risk: 89.7510%
Diagnostic Result: Chronic Kidney Disease DETECTED
